# 面试问题：怎样从零实现 GCN 的归一化消息传递与节点分类？

## 可直接复述的回答主线

1. GCN 先给邻接矩阵加 self-loop，再计算 D^-1/2(A+I)D^-1/2，避免节点度差异让消息尺度失控。
2. 一层传播可写成 H'=ÂHW：先线性变换再按归一化边权聚合邻居与自身信息。
3. 两层 GCN 让每个节点使用两跳结构，适合图上同质性较强的半监督或节点分类任务。
4. self-loop 既保留节点自身特征，也让孤立点度数至少为一，避免除零和 NaN。
5. 评测要在同一节点上比较只看原始特征的 baseline 与 GCN，并展示边、度数、Â、平滑特征、hidden 和逐节点结果。
6. GCN 是 transductive 还是 inductive 取决于训练期可见的节点与图快照，不能只由模型类名判断。
7. 生产还需稀疏算子、采样、动态图版本、跨租户隔离、异配图处理、校准和特征/度数漂移监控。

下面用同一批可读输入依次验证朴素基线、手写核心机制、中间过程、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是 12 个脱敏电商服务节点：6 个用户链路服务和 6 个数据基础设施服务。每个社区内部依赖稠密，只有两条跨社区调用；少数锚点节点有明显类别特征，其余节点原始特征模糊，适合观察图平滑如何传播社区信号。

In [1]:
import math  # 汇总梯度并计算分类指标。
import warnings  # 过滤本地 PyTorch 环境的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦图学习实验。
import torch  # 使用基础矩阵运算和自动微分手写 GCN。
torch.manual_seed(251)  # 固定 GCN 参数初始化与训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程提高可复现性。
node_names = ["搜索页", "商品页", "购物车", "结算页", "优惠券", "客服入口", "画像库", "库存库", "订单库", "日志仓", "风控库", "推荐特征"]  # 定义十二个有业务语义的服务节点。
class_names = ["用户链路", "数据基础设施"]  # 定义两个服务社区标签。
labels = torch.tensor([0] * 6 + [1] * 6, dtype=torch.long)  # 为两个社区分配节点类别。
anchor_signal = [1.2, 0.8, 0.0, 0.0, 0.0, 0.0, -1.2, -0.8, 0.0, 0.0, 0.0, 0.0]  # 只给每个社区两个锚点明显类别信号。
repeated_load = [0.2, 0.4, 0.6, 0.3, 0.5, 0.7] * 2  # 为两类使用完全相同的负载模式。
features = torch.tensor([[anchor_signal[index], repeated_load[index], 1.0] for index in range(12)], dtype=torch.float32)  # 构造锚点、负载和 bias 三维节点特征。
edges = []  # 保存无向服务依赖边。
for offset in [0, 6]:  # 分别构造两个六节点社区。
    for left in range(offset, offset + 6):  # 遍历当前社区左端点。
        for right in range(left + 1, offset + 6):  # 与社区内后续节点形成稠密依赖。
            edges.append((left, right))  # 保存当前无向社区边。
edges.extend([(2, 8), (3, 10)])  # 加入购物车到订单库、结算页到风控库的跨社区调用。
print("教学实验输入：12节点电商服务依赖图")  # 标记下方为可读图结构。
for index, name in enumerate(node_names):  # 逐节点展示特征与类别。
    neighbors = [node_names[right] if left == index else node_names[left] for left, right in edges if left == index or right == index]  # 收集当前服务相邻依赖。
    print(f"{index:02d} {name:<6} class={class_names[labels[index]]:<7} features={features[index].tolist()} neighbors={neighbors}")  # 输出节点字段、真值和邻居列表。
print("edge_count=", len(edges), "cross_edges=", [(node_names[left], node_names[right]) for left, right in edges if labels[left] != labels[right]])  # 展示图规模和少量跨社区边。

教学实验输入：12节点电商服务依赖图
00 搜索页    class=用户链路    features=[1.2000000476837158, 0.20000000298023224, 1.0] neighbors=['商品页', '购物车', '结算页', '优惠券', '客服入口']
01 商品页    class=用户链路    features=[0.800000011920929, 0.4000000059604645, 1.0] neighbors=['搜索页', '购物车', '结算页', '优惠券', '客服入口']
02 购物车    class=用户链路    features=[0.0, 0.6000000238418579, 1.0] neighbors=['搜索页', '商品页', '结算页', '优惠券', '客服入口', '订单库']
03 结算页    class=用户链路    features=[0.0, 0.30000001192092896, 1.0] neighbors=['搜索页', '商品页', '购物车', '优惠券', '客服入口', '风控库']
04 优惠券    class=用户链路    features=[0.0, 0.5, 1.0] neighbors=['搜索页', '商品页', '购物车', '结算页', '客服入口']
05 客服入口   class=用户链路    features=[0.0, 0.699999988079071, 1.0] neighbors=['搜索页', '商品页', '购物车', '结算页', '优惠券']
06 画像库    class=数据基础设施  features=[-1.2000000476837158, 0.20000000298023224, 1.0] neighbors=['库存库', '订单库', '日志仓', '风控库', '推荐特征']
07 库存库    class=数据基础设施  features=[-0.800000011920929, 0.4000000059604645, 1.0] neighbors=['画像库', '订单库', '日志仓', '风控库', '推荐特征']
08 订单库    class=数据基础设施  features=

## 2. Baseline / 基线：原始特征最近类别中心

用同一批十二节点的原始三维特征计算两类中心，再按欧氏距离分类。十个非强锚点的第一维为零，而负载分布在两类中完全相同，因此大量节点并列并被误判为第一类。

In [2]:
class_centroids = torch.stack([features[labels == class_index].mean(dim=0) for class_index in range(2)])  # 计算两类原始特征中心。
baseline_distances = ((features[:, None, :] - class_centroids[None, :, :]) ** 2).sum(dim=2)  # 计算每个节点到两类中心的平方距离。
baseline_predictions = baseline_distances.argmin(dim=1)  # 对并列距离按索引选择用户链路类。
baseline_accuracy = float((baseline_predictions == labels).to(torch.float32).mean().item())  # 计算原始特征基线准确率。
print("Baseline：raw-feature nearest centroid")  # 标记下表为不读取边的方案。
print("node       true          feature                 distances             prediction")  # 输出逐节点基线表头。
for index, name in enumerate(node_names):  # 逐节点展示原始特征判断。
    print(f"{name:<8} {class_names[labels[index]]:<12} {features[index].tolist()!s:<23} {baseline_distances[index].tolist()} {class_names[baseline_predictions[index]]}")  # 输出特征、两类距离和预测。
print(f"Baseline accuracy={baseline_accuracy:.4f}")  # 展示忽略图结构的同数据指标。

Baseline：raw-feature nearest centroid
node       true          feature                 distances             prediction
搜索页      用户链路         [1.2000000476837158, 0.20000000298023224, 1.0] [0.8136111497879028, 2.41361141204834] 用户链路
商品页      用户链路         [0.800000011920929, 0.4000000059604645, 1.0] [0.22027777135372162, 1.2869445085525513] 用户链路
购物车      用户链路         [0.0, 0.6000000238418579, 1.0] [0.13361111283302307, 0.13361111283302307] 用户链路
结算页      用户链路         [0.0, 0.30000001192092896, 1.0] [0.13361111283302307, 0.13361111283302307] 用户链路
优惠券      用户链路         [0.0, 0.5, 1.0]         [0.1136111170053482, 0.1136111170053482] 用户链路
客服入口     用户链路         [0.0, 0.699999988079071, 1.0] [0.1736111044883728, 0.1736111044883728] 用户链路
画像库      数据基础设施       [-1.2000000476837158, 0.20000000298023224, 1.0] [2.41361141204834, 0.8136111497879028] 数据基础设施
库存库      数据基础设施       [-0.800000011920929, 0.4000000059604645, 1.0] [1.2869445085525513, 0.22027777135372162] 数据基础设施
订单库      数据基础设施       [0.0,

## 3. 底层实现：A+I、对称归一化、GCNLayer 与两层 GCN

不使用 PyG、DGL 或现成图卷积层。邻接矩阵、self-loop、度数、Â 和 `Â @ (H @ W)` 都显式构造；第一层返回 support 与 aggregated，便于看到锚点信号如何传播。

In [3]:
def dense_adjacency(node_count, undirected_edges):  # 从无向边列表手写稠密邻接矩阵。
    adjacency = torch.zeros(node_count, node_count, dtype=torch.float32)  # 初始化全零方阵。
    for left, right in undirected_edges:  # 逐边写入两个方向。
        adjacency[left, right] = 1.0  # 写入 left 到 right 连接。
        adjacency[right, left] = 1.0  # 写入 right 到 left 连接。
    return adjacency  # 返回不含 self-loop 的邻接矩阵。
def gcn_normalize(adjacency, add_self_loops=True):  # 手写 GCN 对称归一化。
    adjusted = adjacency + torch.eye(adjacency.shape[0]) if add_self_loops else adjacency.clone()  # 按配置统一加入单位 self-loop。
    degree = adjusted.sum(dim=1)  # 计算每个节点加环后的度数。
    inverse_sqrt_degree = degree.pow(-0.5)  # 计算 D^-1/2 对角元素。
    normalized = inverse_sqrt_degree[:, None] * adjusted * inverse_sqrt_degree[None, :]  # 通过广播得到 D^-1/2 A D^-1/2。
    return normalized, degree  # 返回归一化邻接和度数供解释。
class GCNLayer(torch.nn.Module):  # 定义不调用 nn.Linear 的图卷积层。
    def __init__(self, input_features, output_features):  # 初始化显式 W 与 bias 参数。
        super().__init__()  # 注册可训练张量。
        self.weight = torch.nn.Parameter(torch.empty(input_features, output_features))  # 创建输入维乘输出维权重。
        self.bias = torch.nn.Parameter(torch.zeros(output_features))  # 创建输出通道 bias。
        torch.nn.init.xavier_uniform_(self.weight)  # 用 Xavier 初始化消息变换矩阵。
    def forward(self, node_features, normalized_adjacency, return_debug=False):  # 执行线性变换和图聚合。
        support = node_features @ self.weight  # 先把每个节点映射到输出特征空间。
        aggregated = normalized_adjacency @ support  # 按归一化边权聚合邻居和自身。
        output = aggregated + self.bias  # 为每个节点添加共享 bias。
        return (output, {"support": support, "aggregated": aggregated}) if return_debug else output  # 按需返回消息传递中间量。
class GCN(torch.nn.Module):  # 定义两层节点分类 GCN。
    def __init__(self):  # 初始化三维到八维再到两类的图网络。
        super().__init__()  # 注册两层图卷积参数。
        self.layer_one = GCNLayer(3, 8)  # 创建第一层邻域特征编码。
        self.layer_two = GCNLayer(8, 2)  # 创建第二层类别 logits 输出。
    def forward(self, node_features, normalized_adjacency, return_debug=False):  # 执行两跳消息传递。
        first_output, first_debug = self.layer_one(node_features, normalized_adjacency, return_debug=True)  # 计算第一层 support 和聚合。
        hidden = torch.relu(first_output)  # 对第一层节点表示执行非线性。
        logits, second_debug = self.layer_two(hidden, normalized_adjacency, return_debug=True)  # 再聚合一跳并输出类别分数。
        debug = {"first_support": first_debug["support"], "first_aggregated": first_debug["aggregated"], "hidden": hidden, "second_aggregated": second_debug["aggregated"]}  # 汇总两层消息证据。
        return (logits, debug) if return_debug else logits  # 按需返回中间节点表示。
adjacency = dense_adjacency(len(node_names), edges)  # 构造十二节点不含 self-loop 邻接矩阵。
normalized_adjacency, degrees = gcn_normalize(adjacency)  # 加 self-loop 并执行对称归一化。
smoothed_raw_features = normalized_adjacency @ features  # 直接观察一次无参数图平滑后的三维特征。
model = GCN()  # 创建待训练两层 GCN。
optimizer = torch.optim.Adam(model.parameters(), lr=0.04)  # 创建图节点分类优化器。
history = []  # 保存真实 backward 的损失和梯度轨迹。
for step in range(220):  # 在十二节点受控图上执行全图训练。
    optimizer.zero_grad(set_to_none=True)  # 清除上一步 GCN 参数梯度。
    logits, training_debug = model(features, normalized_adjacency, return_debug=True)  # 前向执行两层图聚合。
    loss = torch.nn.functional.cross_entropy(logits, labels)  # 计算全部十二节点监督交叉熵。
    loss.backward()  # 对两层 W 与 bias 执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总全部非空梯度二范数。
    optimizer.step()  # 应用 Adam 更新图卷积参数。
    if step % 55 == 0 or step == 219:  # 每五十五步保存训练证据。
        accuracy = float((logits.argmax(dim=1) == labels).to(torch.float32).mean().item())  # 计算当前节点分类准确率。
        history.append({"step": step, "loss": loss.item(), "accuracy": accuracy, "gradient_norm": gradient_norm})  # 保存损失、准确率和梯度。
model.eval()  # 切换到确定性图推理模式。
with torch.no_grad():  # 取得最终节点 logits 与中间表示。
    final_logits, final_debug = model(features, normalized_adjacency, return_debug=True)  # 对十二节点执行最终两层传播。
print("GCN训练轨迹=", history)  # 展示损失下降和非零梯度。
print("degree with self-loop=", degrees.tolist())  # 展示社区节点与跨边节点的度数差异。
print("Â左上6x6=", torch.round(normalized_adjacency[:6, :6] * 1000) / 1000)  # 展示用户链路社区的实际归一化边权。
print("raw/smoothed anchor signal=", torch.stack([features[:, 0], smoothed_raw_features[:, 0]], dim=1).tolist())  # 展示锚点信号经一跳平滑向社区传播。
print("first hidden前四节点=", torch.round(final_debug["hidden"][:4] * 1000) / 1000)  # 展示手写第一层聚合后的节点表示。

GCN训练轨迹= [{'step': 0, 'loss': 0.8078597187995911, 'accuracy': 0.5, 'gradient_norm': 0.6009080069798594}, {'step': 55, 'loss': 0.01018926315009594, 'accuracy': 1.0, 'gradient_norm': 0.020278570812963498}, {'step': 110, 'loss': 0.002181270392611623, 'accuracy': 1.0, 'gradient_norm': 0.004643730937994262}, {'step': 165, 'loss': 0.0012652770383283496, 'accuracy': 1.0, 'gradient_norm': 0.0028075191311471817}, {'step': 219, 'loss': 0.0008386070257984102, 'accuracy': 1.0, 'gradient_norm': 0.0019165137807093365}]
degree with self-loop= [6.0, 6.0, 7.0, 7.0, 6.0, 6.0, 6.0, 6.0, 7.0, 6.0, 7.0, 6.0]
Â左上6x6= tensor([[0.1670, 0.1670, 0.1540, 0.1540, 0.1670, 0.1670],
        [0.1670, 0.1670, 0.1540, 0.1540, 0.1670, 0.1670],
        [0.1540, 0.1540, 0.1430, 0.1430, 0.1540, 0.1540],
        [0.1540, 0.1540, 0.1430, 0.1430, 0.1540, 0.1540],
        [0.1670, 0.1670, 0.1540, 0.1540, 0.1670, 0.1670],
        [0.1670, 0.1670, 0.1540, 0.1540, 0.1670, 0.1670]])
raw/smoothed anchor signal= [[1.2000000476837158

## 4. 逐节点分类结果与结果解读

在完全相同的十二节点上比较原始特征中心与 GCN，输出每个服务的预测概率、原始锚点信号和平滑信号，展示图结构带来的实际变化。

In [4]:
final_probabilities = torch.softmax(final_logits, dim=1)  # 把 GCN logits 转为两类概率。
final_predictions = final_probabilities.argmax(dim=1)  # 取得每个节点最高概率类别。
gcn_accuracy = float((final_predictions == labels).to(torch.float32).mean().item())  # 计算 GCN 同数据节点准确率。
print("node       true          raw_base      GCN_pred      confidence  raw_anchor  smoothed_anchor")  # 输出逐节点同数据表头。
for index, name in enumerate(node_names):  # 逐服务展示两种方法和图平滑证据。
    confidence = float(final_probabilities[index, final_predictions[index]].item())  # 读取当前最高类别概率。
    print(f"{name:<8} {class_names[labels[index]]:<12} {class_names[baseline_predictions[index]]:<12} {class_names[final_predictions[index]]:<12} {confidence:>10.4f} {features[index, 0].item():>10.3f} {smoothed_raw_features[index, 0].item():>15.3f}")  # 输出真实类、两种预测与信号传播。
print(f"结果解读：raw-feature centroid accuracy={baseline_accuracy:.4f}，两层手写GCN={gcn_accuracy:.4f}；社区边把少数锚点信号扩散到模糊节点。")  # 解释图结构对相同节点的分类收益。

node       true          raw_base      GCN_pred      confidence  raw_anchor  smoothed_anchor
搜索页      用户链路         用户链路         用户链路             0.9997      1.200           0.333
商品页      用户链路         用户链路         用户链路             0.9997      0.800           0.333
购物车      用户链路         用户链路         用户链路             0.9981      0.000           0.309
结算页      用户链路         用户链路         用户链路             0.9981      0.000           0.309
优惠券      用户链路         用户链路         用户链路             0.9997      0.000           0.333
客服入口     用户链路         用户链路         用户链路             0.9997      0.000           0.333
画像库      数据基础设施       数据基础设施       数据基础设施           0.9997     -1.200          -0.333
库存库      数据基础设施       数据基础设施       数据基础设施           0.9997     -0.800          -0.333
订单库      数据基础设施       用户链路         数据基础设施           0.9982      0.000          -0.309
日志仓      数据基础设施       用户链路         数据基础设施           0.9997      0.000          -0.333
风控库      数据基础设施       用户链路         数据基础设施      

## 5. 失败案例与修正：不加 self-loop 时孤立节点除零

在三节点路径外加入一个孤立服务。若直接对 A 计算 D^-1/2，孤立点度数为零，`0^-0.5` 为 inf，乘回零邻接产生 NaN；统一加 I 后孤立点的归一化 self-loop 为 1。

In [5]:
fixture_adjacency = dense_adjacency(4, [(0, 1), (1, 2)])  # 构造含一个孤立节点的四节点故障图。
bad_degree = fixture_adjacency.sum(dim=1)  # 错误地在不加 self-loop 时计算度数。
bad_inverse_sqrt = bad_degree.pow(-0.5)  # 对孤立点零度数产生正无穷。
bad_normalized = bad_inverse_sqrt[:, None] * fixture_adjacency * bad_inverse_sqrt[None, :]  # 复现零乘无穷产生 NaN 的归一化。
fixed_normalized, fixed_degree = gcn_normalize(fixture_adjacency, add_self_loops=True)  # 用统一 self-loop 修复孤立点度数。
bad_isolated_row = bad_normalized[3]  # 读取错误实现的孤立节点整行。
fixed_isolated_row = fixed_normalized[3]  # 读取修正实现的孤立节点整行。
print(f"错误行为：degree={bad_degree.tolist()}，isolated_row={bad_isolated_row.tolist()}，finite={torch.isfinite(bad_isolated_row).all().item()}")  # 展示零度数导致非有限权重。
print(f"修正行为：degree_with_I={fixed_degree.tolist()}，isolated_row={fixed_isolated_row.tolist()}，finite={torch.isfinite(fixed_isolated_row).all().item()}")  # 展示 self-loop 保留孤立点自身信息。

错误行为：degree=[1.0, 2.0, 1.0, 0.0]，isolated_row=[nan, nan, nan, nan]，finite=False
修正行为：degree_with_I=[2.0, 3.0, 2.0, 1.0]，isolated_row=[0.0, 0.0, 0.0, 1.0]，finite=True


## 6. 生产边界

十二节点稠密矩阵不能代表生产大图。真实系统需 CSR/COO 稀疏算子、邻居或子图采样、快照/特征版本绑定、跨租户边过滤、时间切分、异配图或有向边建模、增量新节点策略、目标硬件吞吐测试，并监控度数分布、孤立点率、特征缺失、校准和逐社区召回率。

In [6]:
gcn_diagnostics = {"nodes": len(node_names), "edges": len(edges), "cross_edges": 2, "baseline_accuracy": baseline_accuracy, "gcn_accuracy": gcn_accuracy, "initial_loss": history[0]["loss"], "final_loss": history[-1]["loss"], "bad_isolated_finite": bool(torch.isfinite(bad_isolated_row).all()), "fixed_isolated_diagonal": float(fixed_isolated_row[3].item())}  # 汇总图、训练、分类和孤立点指标。
print("生产监控快照：", gcn_diagnostics)  # 输出 GCN 节点服务应持续观察的信号。

生产监控快照： {'nodes': 12, 'edges': 32, 'cross_edges': 2, 'baseline_accuracy': 0.6666666865348816, 'gcn_accuracy': 1.0, 'initial_loss': 0.8078597187995911, 'final_loss': 0.0008386070257984102, 'bad_isolated_finite': False, 'fixed_isolated_diagonal': 1.0}


## 7. 最小回归测试

最后一格只保护图规模、归一化、真实训练、分类收益和孤立点修正。

In [7]:
assert len(node_names) >= 6 and len(edges) >= 10 and adjacency.shape == (12, 12)  # 保证案例包含非平凡真实图结构。
assert torch.allclose(normalized_adjacency, normalized_adjacency.T) and torch.isfinite(normalized_adjacency).all()  # 保证 GCN 对称归一化有限正确。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证两层 GCN 真实 backward 学习。
assert gcn_accuracy > baseline_accuracy and gcn_accuracy >= 0.90  # 保证同节点图模型优于原始特征基线。
assert not torch.isfinite(bad_isolated_row).all() and torch.isfinite(fixed_isolated_row).all()  # 保证零度孤立点失败可复现并修正。
assert fixed_degree[3].item() == 1.0 and fixed_isolated_row[3].item() == 1.0  # 保证修正后孤立点只保留单位 self-loop。